# Evaluasi IndoBERT Tanpa Kebocoran Data

Notebook ini melengkapi skripsi dengan evaluasi yang lebih ketat. Hasil skripsi tetap ada di `baseline_indobert.ipynb`.

**Yang diubah:** duplikat dihapus lebih dulu, split dilakukan sebelum augmentasi, data sintetis hanya masuk ke data latih, dan undersampling dilakukan acak.

**Status:** notebook ini **belum dijalankan**, jadi belum ada hasil yang bisa dikutip. Jalankan di Google Colab dengan GPU.

**Cara pakai:** isi kolom `label_manual` di `gold_set_untuk_dilabel.csv` dengan `Negative`, `Neutral`, atau `Positive`, tanpa melihat prediksi model. Skor gold set baru muncul kalau 300 baris sudah terisi.

## 1. Instal dependensi

In [ ]:
!pip install -q datasets evaluate "transformers[torch]" scikit-learn

## 2. Siapkan data

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data") if Path("../data").exists() else Path("/content")
SEED = 42

real = pd.read_csv(DATA_DIR / "data_high_confidence.csv")
aug = pd.read_csv(DATA_DIR / "data_augmentasi.csv")
gold = pd.read_csv(DATA_DIR / "gold_set_untuk_dilabel.csv")


def kunci(s):
    """Samakan teks supaya duplikat dan hampir-duplikat bisa dideteksi."""
    return (s.astype(str).str.lower()
            .str.replace(r"[^a-z0-9 ]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip())


for df in (real, aug, gold):
    df["kunci"] = kunci(df["normalize_text"])

# 1. Hapus duplikat dan keluarkan gold set dari data latih
real = real.drop_duplicates("kunci")
real = real[~real["kunci"].isin(gold["kunci"])]

# 2. Split dulu, sebelum augmentasi dan undersampling
train_df, val_df = train_test_split(
    real, test_size=0.15, stratify=real["label"], random_state=SEED
)

# 3. Augmentasi hanya masuk ke data latih
terlarang = set(val_df["kunci"]) | set(gold["kunci"]) | set(train_df["kunci"])
aug_bersih = aug[~aug["kunci"].isin(terlarang)].drop_duplicates("kunci")
train_df = pd.concat([train_df, aug_bersih], ignore_index=True)

# 4. Undersampling acak, bukan confidence tertinggi
n_min = train_df["label"].value_counts().min()
train_df = (train_df.groupby("label").sample(n_min, random_state=SEED)
            .sample(frac=1, random_state=SEED).reset_index(drop=True))

assert not set(train_df["kunci"]) & set(val_df["kunci"])
assert not set(train_df["kunci"]) & set(gold["kunci"])
print("Train:", train_df["label"].value_counts().to_dict())
print("Validasi:", val_df["label"].value_counts().to_dict())
print("Sintetis di validasi dan gold set: 0")


## 3. Latih model

In [ ]:
import numpy as np
from datasets import Dataset
from sklearn.metrics import classification_report
from transformers import (AutoModelForSequenceClassification, BertTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

MODEL = "indobenchmark/indobert-base-p1"
label2id = {"Negative": 0, "Neutral": 1, "Positive": 2}
id2label = {v: k for k, v in label2id.items()}

tokenizer = BertTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, label2id=label2id, id2label=id2label
)


def siapkan(df):
    ds = Dataset.from_pandas(pd.DataFrame({
        "text": df["normalize_text"].astype(str).tolist(),
        "labels": df["label"].map(label2id).tolist(),
    }))
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128),
                  batched=True, remove_columns=["text"])


trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="results_ketat", learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=5, eval_strategy="epoch", save_strategy="no",
        report_to="none", seed=SEED,
    ),
    train_dataset=siapkan(train_df),
    eval_dataset=siapkan(val_df),
    data_collator=DataCollatorWithPadding(tokenizer),
)
trainer.train()


## 4. Evaluasi

In [ ]:
def laporan(df, judul):
    pred = trainer.predict(siapkan(df)).predictions.argmax(axis=1)
    print(judul)
    print(classification_report(df["label"].map(label2id), pred,
                                target_names=list(label2id), digits=3))


laporan(val_df, "Validasi: tweet asli, label otomatis")

gold_terisi = gold.rename(columns={"label_manual": "label"})
gold_terisi["label"] = gold_terisi["label"].astype("string").str.strip().str.title()
kosong = (~gold_terisi["label"].isin(label2id)).sum()
if kosong:
    print(f"Gold set belum lengkap: {kosong} baris belum dilabel.")
else:
    laporan(gold_terisi, "Gold set: tweet asli, label manual")
